In [11]:
import pandas as pd
import re

# Load files
papers = pd.read_csv("../CsvForDB/paper.csv")
fixed = pd.read_csv("fixed.csv")

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Build a read-only lookup from paper.csv using title + subtitle
papers_review = papers.copy()
papers_review["paper_combined"] = (
    papers_review["title"].fillna("").astype(str) + " " +
    papers_review["subtitle"].fillna("").astype(str)
).str.strip()

papers_review["norm_title"] = papers_review["paper_combined"].apply(normalize)

# Normalize fixed.csv titles
fixed_review = fixed.copy()
fixed_review["norm_title"] = fixed_review["title"].fillna("").astype(str).apply(normalize)

# Prevent empty titles from matching empty titles
paper_titles_set = {x for x in papers_review["norm_title"] if x}
is_match = fixed_review["norm_title"].isin(paper_titles_set) & fixed_review["norm_title"].ne("")

# Split fixed.csv
rdp_pure_match = fixed[is_match].copy()
fixed_remaining = fixed[~is_match].copy()

# Save outputs
rdp_pure_match.to_csv("rdp_pure_match.csv", index=False)
fixed_remaining.to_csv("fixed_remaining.csv", index=False)

# Create side-by-side review file
review_matches = fixed_review[is_match].merge(
    papers_review[["paper_combined", "norm_title"]],
    on="norm_title",
    how="left"
)

review_matches[["title", "paper_combined"]].to_csv("match_review.csv", index=False)

# Optional: flag ambiguous matches where one fixed title matches multiple paper rows
ambiguous_matches = (
    review_matches.groupby("title")
    .size()
    .reset_index(name="n_matches")
)

ambiguous_matches = ambiguous_matches[ambiguous_matches["n_matches"] > 1]
ambiguous_matches.to_csv("ambiguous_matches.csv", index=False)

# Print checks
print("Total fixed:", len(fixed))
print("Matched:", len(rdp_pure_match))
print("Remaining:", len(fixed_remaining))
print("Check sum:", len(rdp_pure_match) + len(fixed_remaining))
print("Review file saved as: match_review.csv")
print("Ambiguous matches saved as: ambiguous_matches.csv")

Total fixed: 32054
Matched: 4574
Remaining: 27480
Check sum: 32054
Review file saved as: match_review.csv
Ambiguous matches saved as: ambiguous_matches.csv


In [12]:
print(pd.read_csv("ambiguous_matches.csv").head(20))
print(pd.read_csv("ambiguous_matches.csv").shape)

                                                title  n_matches
0   " We Would Never Write That Down" Classificati...          2
1   "De har fået NemID, men det er ikke nemt for m...          2
2   "Kinda like The Sims... But with ghosts?" : A ...          2
3   "Kinda like the Sims... but with ghosts?": A Q...          2
4   "My Soul Got a Little Bit Cleaner": Art Experi...          2
5                  4 Types of Business Transformation          2
6   40 Variability Bugs in the Linux Kernel : A Qu...          2
7   42 Variability Bugs in the Linux Kernel : A Qu...          2
8                      A Bestiary of Digital Monsters          5
9   A Case for Declarative Process Modelling: Agil...          2
10  A Constraint Programming Model for Fast Optima...          2
11  A Dataset for the Detection of Dehumanizing La...          2
12              A Fixed-Parameter Perspective on #BIS          2
13  A Linear Time Algorithm for Optimal Quay Crane...          2
14  A Local Perspective: 

In [14]:
review = pd.read_csv("match_review.csv")

review[review["title"] == "A Bestiary of Digital Monsters"]

,title,paper_combined
831,A Bestiary of Digital Monsters,A Bestiary of Digital Monsters
1252,A Bestiary of Digital Monsters,A Bestiary of Digital Monsters
1576,A Bestiary of Digital Monsters,A Bestiary of Digital Monsters
2104,A Bestiary of Digital Monsters,A Bestiary of Digital Monsters
2760,A Bestiary of Digital Monsters,A Bestiary of Digital Monsters


In [15]:
amb = pd.read_csv("ambiguous_matches.csv")

sample_titles = amb.sample(10, random_state=42)["title"]

for t in sample_titles:
    print("\n---", t, "---")
    print(review[review["title"] == t][["title", "paper_combined"]])


--- Effectiveness of job title based embeddings on résumé to job ad recommendation ---
                                                  title  \
387   Effectiveness of job title based embeddings on...   
2405  Effectiveness of job title based embeddings on...   

                                         paper_combined  
387   Effectiveness of job title based embeddings on...  
2405  Effectiveness of job title based embeddings on...  

--- Revisiting Hidden Representations in Transfer Learning for Medical Imaging ---
                                                  title  \
750   Revisiting Hidden Representations in Transfer ...   
2342  Revisiting Hidden Representations in Transfer ...   
4080  Revisiting Hidden Representations in Transfer ...   

                                         paper_combined  
750   Revisiting Hidden Representations in Transfer ...  
2342  Revisiting Hidden Representations in Transfer ...  
4080  Revisiting Hidden Representations in Transfer ...  

--- Di